# Lab 02-02 — Cosine similarity vs raw dot product for retrieval

**Track 02 · Embeddings** — the score your vector store ranks by. Every vector store ranks passages by a similarity score, and the two most common scores are the COSINE SIMILARITY and the raw DOT PRODUCT. This lab answers the question that decides which one your store should use:

    cos(a, b) = (a . b) / (||a|| * ||b||)

The dot product in the numerator is divided by both vector lengths, so cosine measures the ANGLE between two vectors and ignores their magnitude. The raw dot product keeps magnitude: a long vector scores higher than a short one even when it points in a worse direction.

For text embeddings this matters because magnitude is usually noise. BGE (`BAAI/bge-base-en-v1.5`) is trained to produce L2-normalized vectors (`normalize_embeddings=True`), so every vector has length 1 and the two scores collapse into one:

    ||a|| = ||b|| = 1  =>  cos(a, b) = a . b

This notebook is **self-contained**: it imports LangChain, numpy, pandas, and (optionally) scipy directly — no repo component library. The BGE embedder is built right here as a small inline wrapper over `HuggingFaceEmbeddings` with `normalize_embeddings=True`, which is exactly how the shared `src/embeddings/bge.py` component works underneath.

The lab proves that identity numerically on real embeddings, then shows what happens when the normalization is missing: we deliberately scale the passage vectors by a deterministic, length-correlated factor (simulating an embedder that does not normalize, like E5) and watch the raw dot product mis-rank the results that cosine gets right.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-huggingface`, `numpy`, `pandas`, and `scipy` (optional — a pandas fallback computes Spearman if scipy is missing). The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE embeddings
#   langchain-huggingface -> HuggingFaceEmbeddings (the universal embedder class)
#   pandas                -> reads the passages.parquet corpus
#   numpy                 -> score matrices (cosine / dot)
#   scipy                 -> Spearman rank correlation (optional; pandas fallback)
%pip install -q sentence-transformers langchain-huggingface pandas numpy scipy


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

# Silence the "Loading weights" progress bar (transformers honors this flag;
# must be set before any third-party import pulls in huggingface_hub).
import os

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from pathlib import Path

import numpy as np
import pandas as pd

# LangChain + numpy/pandas — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

try:
    from scipy.stats import spearmanr  # noqa: F401

    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False


class BGEEmbedding(Embeddings):
    """Inline BGE wrapper — mirrors src/embeddings/bge.py.

    bge models require normalized embeddings for cosine similarity; the
    universal HuggingFaceEmbeddings class provides that via encode_kwargs.
    """

    def __init__(self, model_name: str = "BAAI/bge-base-en-v1.5"):
        self.model = HuggingFaceEmbeddings(
            model_name=model_name,
            encode_kwargs={"normalize_embeddings": True},
        )

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(text)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.embed_documents(texts)


# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `PASSAGES_PATH` and `TEST_PATH` point at the rag-mini-wikipedia parquet files already on disk; `N_PASSAGES = 50` takes a deterministic head of the 3200-passage corpus; `N_QUESTIONS = 5` the deterministic head of `test.parquet`; `TOP_K = 5` is the retrieval depth, `PREVIEW` truncates the passage previews, and `MODEL_NAME` names the local BGE model.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
MODEL_NAME = "BAAI/bge-base-en-v1.5"
N_PASSAGES = 50  # deterministic head of the 3200-passage corpus (keeps runtime low)
N_QUESTIONS = 5  # deterministic head of test.parquet
TOP_K = 5
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` passages as `(passage_texts, passage_ids)` — the ids are the parquet row indices. `load_questions` returns the first `n` question strings from `test.parquet`.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, n: int) -> list[str]:
    """Return the first ``n`` question strings from test.parquet."""
    df = pd.read_parquet(path)
    return df["question"].head(n).tolist()


## 3. Score matrices — cosine, normalized dot, and deliberately raw dot

`cosine_similarity_matrix` writes the formula out explicitly (rather than via sklearn) so it is visible: each row/column is divided by its L2 norm before the dot product. `normalized_dot_matrix` is the raw dot product, valid because BGE already L2-normalizes every vector — if the embedder's contract holds (`||a|| = ||b|| = 1`), it must equal the cosine matrix above. `unnormalized_passages` simulates an embedder that does NOT normalize (e.g. E5): each passage vector is scaled by a deterministic factor derived from its character length, so the raw dot product then ranks by magnitude * direction instead of direction alone. `raw_dot_matrix` scores against those deliberately unnormalized vectors.


In [ ]:
# --------------------------------------------------------------------------
# 3. Score matrices — cosine, normalized dot, and deliberately raw dot
# --------------------------------------------------------------------------
def cosine_similarity_matrix(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    """cos(a, b) = (a . b) / (||a|| * ||b||) for every (query, passage) pair.

    Written out explicitly (rather than via sklearn) so the formula is
    visible: each row/column is divided by its L2 norm before the dot product.
    """
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    Pn = P / np.linalg.norm(P, axis=1, keepdims=True)
    return Qn @ Pn.T


def normalized_dot_matrix(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    """Raw dot product, valid because BGE already L2-normalizes every vector.

    If the embedder's contract holds (||a|| = ||b|| = 1), this must equal the
    cosine matrix above — the identity this lab proves numerically.
    """
    return Q @ P.T


def unnormalized_passages(P: np.ndarray, lengths: list[int]) -> tuple[np.ndarray, np.ndarray]:
    """Simulate an embedder that does NOT normalize its output.

    BGE normalizes every vector to unit length, so magnitude carries no
    signal. Real embedders that skip normalization (e.g. E5) produce vectors
    whose magnitude grows with passage length. We reproduce that effect by
    scaling each passage vector by a deterministic factor derived from its
    character length — the raw dot product then ranks by magnitude * direction
    instead of direction alone. Returns (scaled_vectors, scale_factors).
    """
    scale = np.asarray([1.0 + (length % 200) / 100.0 for length in lengths])
    return P * scale[:, np.newaxis], scale


def raw_dot_matrix(Q: np.ndarray, P_raw: np.ndarray) -> np.ndarray:
    """Raw dot product on the deliberately unnormalized passage vectors."""
    return Q @ P_raw.T


## 4. Ranking helpers

`top_k_indices` returns the indices of the top-k passages for one query's score row. `spearman_between` computes the Spearman rank correlation between two score rows (via scipy when present, pandas fallback otherwise). `topk_overlap` is the order-insensitive fraction of top-k items shared by two orderings, and `preview` flattens a passage onto one line for printing.


In [ ]:
# --------------------------------------------------------------------------
# 4. Ranking helpers
# --------------------------------------------------------------------------
def top_k_indices(scores: np.ndarray, k: int) -> list[int]:
    """Indices of the top-k passages for one query's score row."""
    return np.argsort(scores)[::-1][:k].tolist()


def spearman_between(cos_scores: np.ndarray, raw_scores: np.ndarray) -> float:
    """Spearman rank correlation between two score rows (per query)."""
    cos_ranks = np.argsort(np.argsort(cos_scores))
    raw_ranks = np.argsort(np.argsort(raw_scores))
    if HAVE_SCIPY:
        return float(spearmanr(cos_ranks, raw_ranks).statistic)
    return float(pd.Series(cos_ranks).corr(pd.Series(raw_ranks), method="spearman"))


def topk_overlap(a: list[int], b: list[int]) -> float:
    """Fraction of top-k items shared by two orderings (order-insensitive)."""
    return len(set(a) & set(b)) / len(a)


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 5. Run the experiment — embed, score three ways, measure the disagreement

`run_experiment` embeds the 50 passages and 5 questions once with BGE, builds the three score matrices, and measures everything the demo prints: the identity gap `max |cosine - normalized_dot|`, per-query Spearman / top-`TOP_K` overlap / identical-order flags between cosine and raw dot, and the concrete mis-ranking example (the first query whose top-1 differs, with its top-`TOP_K` under all three scorers). Everything is returned in one dict — no printing happens here.


In [ ]:
# --------------------------------------------------------------------------
# 5. Run the experiment — embed, score all three, measure the disagreement
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, N_QUESTIONS)

    embedder = BGEEmbedding(model_name=MODEL_NAME)
    P = np.asarray(embedder.embed_documents(passage_texts), dtype=np.float32)
    Q = np.asarray([embedder.embed_query(q) for q in questions], dtype=np.float32)

    cosine = cosine_similarity_matrix(Q, P)
    norm_dot = normalized_dot_matrix(Q, P)
    P_raw, scale = unnormalized_passages(P, [len(t) for t in passage_texts])
    raw_dot = raw_dot_matrix(Q, P_raw)

    max_diff = float(np.max(np.abs(cosine - norm_dot)))

    rho_list, overlap_list, identical_list = [], [], []
    cos_top5, norm_top5, raw_top5 = [], [], []
    for i in range(len(questions)):
        cos_top = top_k_indices(cosine[i], TOP_K)
        norm_top = top_k_indices(norm_dot[i], TOP_K)
        raw_top = top_k_indices(raw_dot[i], TOP_K)
        cos_top5.append(cos_top)
        norm_top5.append(norm_top)
        raw_top5.append(raw_top)
        rho_list.append(spearman_between(cosine[i], raw_dot[i]))
        overlap_list.append(topk_overlap(cos_top, raw_top))
        identical_list.append(cos_top == raw_top)

    demo = next(
        (
            i
            for i in range(len(questions))
            if top_k_indices(cosine[i], 1) != top_k_indices(raw_dot[i], 1)
        ),
        0,
    )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "P": P,
        "Q": Q,
        "cosine": cosine,
        "norm_dot": norm_dot,
        "raw_dot": raw_dot,
        "scale": scale,
        "max_diff": max_diff,
        "rho_list": rho_list,
        "overlap_list": overlap_list,
        "identical_list": identical_list,
        "cos_top5": cos_top5,
        "norm_top5": norm_top5,
        "raw_top5": raw_top5,
        "demo": demo,
    }


## 6. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the setup (corpus subset, questions, Spearman backend); the identity proof — `max |cosine - normalized_dot|` over all scores, which must be ~0 for a normalized embedder; the disagreement — mean Spearman, mean top-`TOP_K` overlap, and how many queries keep an identical top-`TOP_K` under raw dot, plus the concrete mis-ranking example with the three scorers side by side; and a takeaway on when a dot-product store is safe.


In [ ]:
# --------------------------------------------------------------------------
# 6. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    passage_texts, passage_ids = exp["passage_texts"], exp["passage_ids"]
    questions = exp["questions"]
    cosine, norm_dot, raw_dot = exp["cosine"], exp["norm_dot"], exp["raw_dot"]
    scale = exp["scale"]

    print("=" * 66)
    print("Lab 02 — cosine similarity vs raw dot product for retrieval")
    print(f"model: {MODEL_NAME} (local BGE, L2-normalized embeddings)")
    print("=" * 66)

    print(f"\n[1] Setup")
    print(f"    {len(passage_texts)} passages (first {N_PASSAGES} of 3200, "
          f"ids {passage_ids[0]}..{passage_ids[-1]})")
    print(f"    {len(questions)} questions from test.parquet (first {N_QUESTIONS}):")
    for q in questions:
        print(f"      - {q}")
    print(f"    Spearman via {'scipy' if HAVE_SCIPY else 'pandas fallback'}")

    print(f"\n    embedded {exp['P'].shape[0]} passages x {exp['P'].shape[1]} dims, "
          f"{exp['Q'].shape[0]} queries x {exp['Q'].shape[1]} dims")

    print("\n[2] Cosine vs normalized dot — the identity proof")
    print("    cos(a,b) = (a.b)/(||a||*||b||); BGE guarantees ||a||=||b||=1,")
    print("    so cos(a,b) = a.b. The two matrices must be numerically equal:")
    print(f"    max |cosine - normalized_dot| over all {cosine.size} scores = {exp['max_diff']:.3e}")
    print("    -> identical to within float32 precision. For a normalized")
    print("       embedder, cosine similarity IS the dot product.")

    print("\n[3] Cosine vs raw dot on deliberately unnormalized vectors")
    print("    Passage vectors were scaled by a deterministic length-correlated")
    print("    factor (1.0..2.99) to simulate an embedder that skips L2")
    print("    normalization. Raw dot then ranks by magnitude * direction.")
    print(f"    Spearman(cosine ranks, raw-dot ranks), mean over "
          f"{len(questions)} queries: {np.mean(exp['rho_list']):.3f}")
    print(f"    top-{TOP_K} overlap (shared items, order-insensitive), mean: "
          f"{np.mean(exp['overlap_list']):.2f}")
    print(f"    queries where the top-{TOP_K} order is identical: "
          f"{sum(exp['identical_list'])}/{len(questions)}")

    demo = exp["demo"]
    cos_top = exp["cos_top5"][demo]
    norm_top = exp["norm_top5"][demo]
    raw_top = exp["raw_top5"][demo]
    print(f"\n    Concrete example — question {demo}:")
    print(f'      "{questions[demo]}"')
    print(f"      {'rank':<5}{'cosine':>8}{'norm_dot':>10}{'raw_dot':>8}"
          f"   (passage ids into the {N_PASSAGES}-passage subset)")
    for r in range(TOP_K):
        print(f"      {r + 1:<5}{cos_top[r]:>8}{norm_top[r]:>10}{raw_top[r]:>8}")
    print("    cosine and normalized_dot return the identical top-5; raw_dot")
    print("    reorders it. The raw-dot winner is a longer passage whose")
    print(f"    magnitude factor {scale[raw_top[0]]:.2f} outweighs its worse direction:")
    print(f"      raw_dot #1 (id {raw_top[0]}): {preview(passage_texts[raw_top[0]])}")
    print(f"      cosine  #1 (id {cos_top[0]}): {preview(passage_texts[cos_top[0]])}")

    print("\n[4] Takeaway")
    print("    Cosine similarity is the standard for text embeddings because")
    print("    it isolates direction (meaning) from magnitude (length), which")
    print("    is usually noise. The raw dot product is only equivalent when")
    print("    the embedder guarantees unit-norm vectors — BGE does, so a")
    print("    dot-product vector store is safe with BGE. If you swap in an")
    print("    embedder that does not normalize (e.g. E5), either normalize")
    print("    the vectors yourself or switch the store's metric to cosine.")


## 7. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: the matrices have the right shapes; the identity holds — `max |cosine - normalized_dot| < 1e-4` and cosine top-`TOP_K` equals normalized-dot top-`TOP_K` for every query; the scale factors really simulate an unnormalized embedder (all in `[1.0, 2.99)`); and the raw dot actually mis-ranks — mean top-`TOP_K` overlap below 1.0, mean Spearman below 1.0, and the demo query's raw-dot top-1 differs from its cosine top-1. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 7. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append((f"passage matrix is {N_PASSAGES} x 768",
                   exp["P"].shape == (N_PASSAGES, 768)))
    checks.append((f"query matrix is {N_QUESTIONS} x 768",
                   exp["Q"].shape == (N_QUESTIONS, 768)))
    checks.append(("cosine scores all in [-1, 1]",
                   bool(np.all((exp["cosine"] >= -1.0) & (exp["cosine"] <= 1.0)))))
    checks.append((f"identity: max |cosine - normalized_dot| = {exp['max_diff']:.2e} < 1e-4",
                   exp["max_diff"] < 1e-4))
    for i in range(N_QUESTIONS):
        checks.append((f"Q{i}: cosine top-{TOP_K} == normalized-dot top-{TOP_K}",
                       exp["cos_top5"][i] == exp["norm_top5"][i]))
    checks.append(("scale factors simulate an unnormalized embedder (1.0..2.99)",
                   all(1.0 <= s < 3.0 for s in exp["scale"])))
    checks.append((f"raw dot reorders results: mean top-{TOP_K} overlap "
                   f"{np.mean(exp['overlap_list']):.2f} < 1.0",
                   np.mean(exp["overlap_list"]) < 1.0))
    checks.append((f"raw dot reorders results: mean Spearman "
                   f"{np.mean(exp['rho_list']):.3f} < 1.0",
                   np.mean(exp["rho_list"]) < 1.0))
    checks.append(("demo row exists: raw-dot top-1 != cosine top-1",
                   exp["raw_top5"][exp["demo"]][0] != exp["cos_top5"][exp["demo"]][0]))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute of local embedding (BGE, cached on disk) — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The identity proof and the disagreement: cosine vs normalized dot must be numerically identical, while the deliberately unnormalized vectors make the raw dot product mis-rank — with the concrete example printed side by side.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
